In [10]:
!pip install faker

import sqlite3
import csv
import zipfile
from faker import Faker
import random

f = Faker('fr_FR')

con = sqlite3.connect(':memory:')
cur = con.cursor()

ldd = """
DROP TABLE IF EXISTS compose;
DROP TABLE IF EXISTS constitue2;
DROP TABLE IF EXISTS constitue;
DROP TABLE IF EXISTS est_composee;
DROP TABLE IF EXISTS participe;
DROP TABLE IF EXISTS REUNION;
DROP TABLE IF EXISTS CLUB;
DROP TABLE IF EXISTS ENTRETIEN;
DROP TABLE IF EXISTS MACHINE;
DROP TABLE IF EXISTS PLAT;
DROP TABLE IF EXISTS LIEU;
DROP TABLE IF EXISTS ORDRE;
DROP TABLE IF EXISTS TENRAC;
DROP TABLE IF EXISTS INGREDIENT;
DROP TABLE IF EXISTS MODELE;
DROP TABLE IF EXISTS SAUCE;
DROP TABLE IF EXISTS LEGUME;
DROP TABLE IF EXISTS ALIMENT;
DROP TABLE IF EXISTS TERRITOIRE;
DROP TABLE IF EXISTS ORGANISATION;
DROP TABLE IF EXISTS GRADE_SUIVANT;
DROP TABLE IF EXISTS GRADE;
DROP TABLE IF EXISTS DIGNITE;
DROP TABLE IF EXISTS TITRE;
DROP TABLE IF EXISTS RANG;
DROP TABLE IF EXISTS ORGANISME;

CREATE TABLE ORGANISME(
   siret INT,
   nomOrganisme VARCHAR(100),
   raisonSociale TEXT,
   PRIMARY KEY(siret)
);

CREATE TABLE RANG(
   nomRang VARCHAR(50),
   PRIMARY KEY(nomRang),
   CONSTRAINT chk_rang CHECK (nomRang IN ('Novice', 'Compagnon'))
);

CREATE TABLE TITRE(
   nomTitre VARCHAR(50),
   PRIMARY KEY(nomTitre),
   CONSTRAINT chk_titre CHECK (nomTitre IN ('Philanthrope', 'Protecteur', 'Honorable'))
);

CREATE TABLE DIGNITE(
   nomDignite VARCHAR(50),
   PRIMARY KEY(nomDignite),
   CONSTRAINT chk_dignite CHECK (nomDignite IN ('Maître', 'Grand Chancelier', 'Grand Maître'))
);

CREATE TABLE GRADE(
   nomGrade VARCHAR(50),
   PRIMARY KEY(nomGrade),
   CONSTRAINT chk_grade CHECK (nomGrade IN ('Affilié', 'Sympathisant', 'Adhérent', 'Chevalier / Dame', 'Grand Chevalier / Haute Dame', 'Commandeur', 'Grand''Croix'))
);

CREATE TABLE GRADE_SUIVANT(
   nomGrade VARCHAR(50),
   nomGradSuiv VARCHAR(50),
   PRIMARY KEY(nomGrade, nomGradSuiv),
   UNIQUE(nomGrade),
   FOREIGN KEY(nomGrade) REFERENCES GRADE(nomGrade)
);

CREATE TABLE ORGANISATION(
   numOrganisation INT,
   nomOrganisation VARCHAR(100),
   PRIMARY KEY(numOrganisation)
);

CREATE TABLE TERRITOIRE(
   idTerritoire INT,
   nomTerritoire VARCHAR(100),
   PRIMARY KEY(idTerritoire)
);

CREATE TABLE ALIMENT(
   nomA VARCHAR(100),
   PRIMARY KEY(nomA)
);

CREATE TABLE LEGUME(
   nomLegume VARCHAR(100),
   PRIMARY KEY(nomLegume)
);

CREATE TABLE SAUCE(
   nomSauce VARCHAR(100),
   PRIMARY KEY(nomSauce)
);

CREATE TABLE MODELE(
   nomModele VARCHAR(100),
   typeEntretien VARCHAR(100),
   periodiciteEntretien TEXT,
   PRIMARY KEY(nomModele)
);

CREATE TABLE INGREDIENT(
   nomIngredient VARCHAR(100),
   PRIMARY KEY(nomIngredient)
);

CREATE TABLE TENRAC(
   codeTenrac INT,
   nomTenrac VARCHAR(100),
   courrielT VARCHAR(150),
   telT VARCHAR(20),
   adresseT VARCHAR(255),
   numOrganisation INT NOT NULL,
   nomGrade VARCHAR(50) NOT NULL,
   nomRang VARCHAR(50),
   nomTitre VARCHAR(50),
   nomDignite VARCHAR(50),
   siret INT NOT NULL,
   PRIMARY KEY(codeTenrac),
   FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation),
   FOREIGN KEY(nomGrade) REFERENCES GRADE(nomGrade),
   FOREIGN KEY(nomRang) REFERENCES RANG(nomRang),
   FOREIGN KEY(nomTitre) REFERENCES TITRE(nomTitre),
   FOREIGN KEY(nomDignite) REFERENCES DIGNITE(nomDignite),
   FOREIGN KEY(siret) REFERENCES ORGANISME(siret)
);

CREATE TABLE ORDRE(
   numOrganisation INT,
   idTerritoire INT NOT NULL,
   PRIMARY KEY(numOrganisation),
   UNIQUE(idTerritoire),
   FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation),
   FOREIGN KEY(idTerritoire) REFERENCES TERRITOIRE(idTerritoire)
);

CREATE TABLE LIEU(
   idLieu INT,
   adresseLieu VARCHAR(255),
   numOrganisation INT NOT NULL,
   PRIMARY KEY(idLieu),
   FOREIGN KEY(numOrganisation) REFERENCES ORDRE(numOrganisation)
);

CREATE TABLE PLAT(
   idPlat INT,
   nomLegume VARCHAR(100),
   PRIMARY KEY(idPlat),
   FOREIGN KEY(nomLegume) REFERENCES LEGUME(nomLegume)
);

CREATE TABLE MACHINE(
   nomMachine VARCHAR(100),
   nomModele VARCHAR(100) NOT NULL,
   PRIMARY KEY(nomMachine),
   FOREIGN KEY(nomModele) REFERENCES MODELE(nomModele)
);

CREATE TABLE ENTRETIEN(
   idEntretien INT,
   date_ DATE,
   nomMachine VARCHAR(100) NOT NULL,
   numOrganisation INT NOT NULL,
   codeTenrac INT NOT NULL,
   PRIMARY KEY(idEntretien),
   FOREIGN KEY(nomMachine) REFERENCES MACHINE(nomMachine),
   FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation),
   FOREIGN KEY(codeTenrac) REFERENCES TENRAC(codeTenrac)
);

CREATE TABLE CLUB(
   numOrganisation_1 INT,
   numOrganisation INT,
   PRIMARY KEY(numOrganisation_1),
   FOREIGN KEY(numOrganisation_1) REFERENCES ORGANISATION(numOrganisation),
   FOREIGN KEY(numOrganisation) REFERENCES ORDRE(numOrganisation)
);

CREATE TABLE REUNION(
   idReunion INT,
   dateReunion DATE,
   intitule TEXT,
   nomMachine VARCHAR(100) NOT NULL,
   idLieu INT NOT NULL,
   PRIMARY KEY(idReunion),
   FOREIGN KEY(nomMachine) REFERENCES MACHINE(nomMachine),
   FOREIGN KEY(idLieu) REFERENCES LIEU(idLieu)
);

CREATE TABLE participe(
   codeTenrac INT,
   idReunion INT,
   PRIMARY KEY(codeTenrac, idReunion),
   FOREIGN KEY(codeTenrac) REFERENCES TENRAC(codeTenrac),
   FOREIGN KEY(idReunion) REFERENCES REUNION(idReunion)
);

CREATE TABLE est_composee(
   idReunion INT,
   idPlat INT,
   PRIMARY KEY(idReunion, idPlat),
   FOREIGN KEY(idReunion) REFERENCES REUNION(idReunion),
   FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat)
);

CREATE TABLE constitue(
   idPlat INT,
   nomA VARCHAR(100),
   PRIMARY KEY(idPlat, nomA),
   FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat),
   FOREIGN KEY(nomA) REFERENCES ALIMENT(nomA)
);

CREATE TABLE constitue2(
   idPlat INT,
   nomSauce VARCHAR(100),
   PRIMARY KEY(idPlat, nomSauce),
   FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat),
   FOREIGN KEY(nomSauce) REFERENCES SAUCE(nomSauce)
);

CREATE TABLE compose(
   nomSauce VARCHAR(100),
   nomIngredient VARCHAR(100),
   PRIMARY KEY(nomSauce, nomIngredient),
   FOREIGN KEY(nomSauce) REFERENCES SAUCE(nomSauce),
   FOREIGN KEY(nomIngredient) REFERENCES INGREDIENT(nomIngredient)
);
"""
cur.executescript(ldd)

grades = ['Affilié', 'Sympathisant', 'Adhérent', 'Chevalier / Dame', 'Grand Chevalier / Haute Dame', 'Commandeur', "Grand'Croix"]
poids_grades = [45, 25, 15, 8, 4, 2, 1]

rangs = ['Novice', 'Compagnon', None]
poids_rangs = [15, 5, 80]

titres = ['Philanthrope', 'Protecteur', 'Honorable', None]
poids_titres = [4, 4, 2, 90]

dignites = ['Maître', 'Grand Chancelier', 'Grand Maître', None]
poids_dignites = [2, 1, 1, 96]

cur.executemany("INSERT INTO GRADE VALUES (?)", [(g,) for g in grades])
cur.executemany("INSERT INTO RANG VALUES (?)", [(r,) for r in rangs if r is not None])
cur.executemany("INSERT INTO TITRE VALUES (?)", [(t,) for t in titres if t is not None])
cur.executemany("INSERT INTO DIGNITE VALUES (?)", [(d,) for d in dignites if d is not None])

sirets = [10000000000000 + i for i in range(1, 21)]
for s in sirets:
    cur.execute("INSERT INTO ORGANISME VALUES (?, ?, ?)", (s, f.company(), f.catch_phrase()))

regions = ['Île-de-France', 'Auvergne-Rhône-Alpes', 'Nouvelle-Aquitaine', 'Occitanie', 'Hauts-de-France', 'Grand Est', 'Bretagne', 'Pays de la Loire', 'Normandie', 'Bourgogne-Franche-Comté', 'Centre-Val de Loire', 'Corse', 'Provence-Alpes-Côte d''Azur']
for i, reg in enumerate(regions, 1):
    cur.execute("INSERT INTO TERRITOIRE VALUES (?, ?)", (i, reg))

for i in range(1, 14):
    cur.execute("INSERT INTO ORGANISATION VALUES (?, ?)", (i, f"Ordre_de_{regions[i-1].replace(' ', '_')}"))
    cur.execute("INSERT INTO ORDRE VALUES (?, ?)", (i, i))

for i in range(14, 101):
    cur.execute("INSERT INTO ORGANISATION VALUES (?, ?)", (i, f"Club_Raclette_{i}"))
    ordre_parent = random.randint(1, 13)
    cur.execute("INSERT INTO CLUB VALUES (?, ?)", (i, ordre_parent))

org_ids = list(range(1, 101))
poids_orgs = [random.randint(10, 1000) for _ in org_ids]

liste_tenracs = []
for i in range(1, 300001):
    rang_choisi = random.choices(rangs, weights=poids_rangs, k=1)[0]
    titre_choisi = random.choices(titres, weights=poids_titres, k=1)[0]
    dignite_choisie = random.choices(dignites, weights=poids_dignites, k=1)[0]
    grade_choisi = random.choices(grades, weights=poids_grades, k=1)[0]
    org_choisie = random.choices(org_ids, weights=poids_orgs, k=1)[0]

    liste_tenracs.append((
        i,
        f.name(),
        f.email(),
        "0600000000",
        f.address().replace('\n', ' '),
        org_choisie,
        grade_choisi,
        rang_choisi,
        titre_choisi,
        dignite_choisie,
        random.choice(sirets)
    ))

    if i % 100000 == 0:
        cur.executemany("INSERT INTO TENRAC VALUES (?,?,?,?,?,?,?,?,?,?,?)", liste_tenracs)
        liste_tenracs = []

with open('SAE_Base_Tenrac.sql', 'w', encoding='utf-8') as f_sql:
    f_sql.write("BEGIN TRANSACTION;\n")
    f_sql.write(ldd + "\n")
    for line in con.iterdump():
        if line.startswith('INSERT INTO'):
            f_sql.write('%s\n' % line)
    f_sql.write("COMMIT;\n")

with zipfile.ZipFile('SAE_SQL_Complet.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('SAE_Base_Tenrac.sql')

cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cur.fetchall()

fichiers_csv_crees = []

for table_name in tables:
    nom_table = table_name[0]

    if nom_table.startswith('sqlite_'):
        continue

    cur.execute(f"SELECT * FROM {nom_table}")

    nom_fichier_csv = f"{nom_table}.csv"
    with open(nom_fichier_csv, 'w', newline='', encoding='utf-8') as f_csv:
        writer = csv.writer(f_csv)

        if cur.description:
            en_tetes = [description[0] for description in cur.description]
            writer.writerow(en_tetes)
            writer.writerows(cur)

    fichiers_csv_crees.append(nom_fichier_csv)

with zipfile.ZipFile('SAE_CSV_Complet.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fichier in fichiers_csv_crees:
        zipf.write(fichier)

con.close()